# TCC · EDA de Marketing Mix Modeling com Google Meridian

**Dados simulados oficiais, foco GEO × TEMPO.** Não há estimação posterior, ROI ou recomendação de orçamento.

Execute de cima para baixo em runtime novo Python 3.12+ (CPU). A primeira execução instala dependências; isso pode levar alguns minutos. Se o Colab já tinha versões diferentes de NumPy/TensorFlow carregadas, reinicie a sessão após instalar e execute novamente.

O bootstrap clona o repositório público quando necessário, preserva instalações compatíveis e salva tudo na pasta do projeto. A fonte de dados e a API estão fixadas por commit/checksum. Para uma cópia histórica exata do projeto, configure `REPO_REF` com o commit da entrega. Os dados não possuem nomes de canais reais ou localização geográfica real.


In [1]:
# Execute esta célula primeiro. Em Colab novo, usa somente CPU.
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/thiagossilva17/MMM_Google_Meridian_Project.git"
REPO_REF = "main"  # Para repetir uma execução publicada, substitua pelo SHA do commit.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src/config.py").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd / "MMM_Google_Meridian_Project"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_REF], check=True)
    elif not (PROJECT_ROOT / "src/config.py").exists():
        raise RuntimeError("Diretório de destino existe, mas não contém o projeto esperado.")

required = ["meridian", "pandas", "numpy", "scipy", "matplotlib", "statsmodels", "tabulate"]
missing = any(importlib.util.find_spec(package) is None for package in required)
try:
    wrong_meridian = importlib.metadata.version("google-meridian") != "2.1.0"
except importlib.metadata.PackageNotFoundError:
    wrong_meridian = True
if missing or wrong_meridian:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements-colab.txt")], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs/.mplcache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from src.config import SEED, ROOT
from src.data import load_panel
from src.reporting import show
import numpy as np
np.random.seed(SEED)
geo_time_panel = load_panel()
print(f"Colab: {IN_COLAB}; projeto: {ROOT}; painel: {geo_time_panel.shape}")


Colab: False; projeto: /workspace/scratch/5d823022ed95/tcc-meridian; painel: (6240, 19)


## 00 — O que recebemos e podemos confiar na estrutura?

### Pergunta
O que recebemos e podemos confiar na estrutura?

### Motivação
Antes de procurar associações, é necessário verificar se cada linha representa um geo em uma semana, sem duplicação, lacunas ou valores impossíveis.

### Método
Verificar SHA-256, schema documentado no exemplo oficial, índice exportado, chave GEO × tempo, grade de 7 dias, nulos, domínios, população e pares gasto/impressões. Calcular CPMU com denominador zero indefinido.

### Interpretação antes de olhar os resultados
Painel completo permite as decomposições balanceadas posteriores. Controles negativos não são automaticamente inválidos: são índices sintéticos. Conversões não inteiras não devem ser arredondadas. Problemas de contrato interrompem a execução após salvar a auditoria.

**Dependências:** painel validado e funções em `src/eda.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [2]:
from src.eda import data_audit
report_00 = data_audit(geo_time_panel)
show(report_00)

### Resultado observado e conclusão parcial

Recebemos 6,240 registros, 40 geos e 156 semanas, de 2021-01-25 a 2024-01-15. A grade é semanal, completa e sem duplicatas, nulos ou violações de domínio verificadas. População é constante dentro de cada geo.

O CSV original tem uma coluna de índice exportado, validada como 0…N−1 e removida apenas da cópia processada. Conversões são valores contínuos simulados; controles podem ser negativos na escala fornecida. Há 5 canais pagos, uma mídia orgânica e Promo. Não há reach/frequency; nenhuma variável desse tipo foi inventada.

CPMU = gasto/impressões. 7496 razões são indefinidas por ausência de impressões e permanecem NaN. Conferir os contadores de inconsistências na auditoria. Isso distingue zero atividade de custo zero.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Distribuições, concentração de investimento, zeros e extremos.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**data_audit** — 29 linhas; CSV completo em `outputs/tables/`.

,metric,value
0,rows,6240
1,geos,40
2,periods,156
3,expected_rows,6240
4,unique_geo_time,6240
5,panel_completeness,1.0
6,duplicate_keys,0
7,missing_geo_time,0
8,missing_cells,0
9,nonfinite_numeric,0


**dictionary** — 19 linhas; CSV completo em `outputs/tables/`.

,variable,role,dtype,dimension,unit,missing,zeros,min,max,mean,std
0,geo,geo,object,geo × time,geo,0,NaN,Geo0,Geo9,NaN,NaN
1,time,time,datetime64[ns],geo × time,time,0,NaN,2021-01-25 00:00:00,2024-01-15 00:00:00,NaN,NaN
2,Channel0_impression,media,int64,geo × time,impressões,0,755.0,0,5192032,8.851635e+05,8.192795e+05
3,Channel1_impression,media,int64,geo × time,impressões,0,1742.0,0,4397540,5.206055e+05,6.205734e+05
4,Channel2_impression,media,int64,geo × time,impressões,0,4010.0,0,5610156,2.605050e+05,5.551397e+05
5,Channel3_impression,media,int64,geo × time,impressões,0,169.0,0,7635147,1.806810e+06,1.304714e+06
6,Channel4_impression,media,int64,geo × time,impressões,0,820.0,0,6975542,9.816780e+05,9.141036e+05
7,competitor_sales_control,control,float64,geo × time,escala sintética; unidade não especificada,0,0.0,-4.825634,3.924682,-4.441200e-02,1.210067e+00
8,sentiment_score_control,control,float64,geo × time,escala sintética; unidade não especificada,0,0.0,-4.230392,4.320259,-3.367510e-02,1.175478e+00
9,Channel0_spend,spend,float64,geo × time,unidade monetária não especificada,0,755.0,0.0,38071.73,6.490659e+03,6.007550e+03


**CPMU** — 5 linhas; CSV completo em `outputs/tables/`.

,channel,undefined,n,mean,median,std,variance,min,max,p01,p05,p25,p50,p75,p95,p99,iqr,cv,zeros,zero_share
0,Channel0,755,5485,0.007333,0.007333,2.433514e-10,5.921988e-20,0.007333,0.007333,0.007333,0.007333,0.007333,0.007333,0.007333,0.007333,0.007333,3.470033e-10,3.318704e-08,0,0.0
1,Channel1,1742,4498,0.009641,0.009641,3.226900e-10,1.041288e-19,0.009641,0.009641,0.009641,0.009641,0.009641,0.009641,0.009641,0.009641,0.009641,4.495688e-10,3.346979e-08,0,0.0
2,Channel2,4010,2230,0.007431,0.007431,2.582488e-10,6.669246e-20,0.007431,0.007431,0.007431,0.007431,0.007431,0.007431,0.007431,0.007431,0.007431,3.643462e-10,3.475326e-08,0,0.0
3,Channel3,169,6071,0.007793,0.007793,2.581763e-10,6.665502e-20,0.007793,0.007793,0.007793,0.007793,0.007793,0.007793,0.007793,0.007793,0.007793,3.653120e-10,3.312994e-08,0,0.0
4,Channel4,820,5420,0.007792,0.007792,2.578769e-10,6.650050e-20,0.007792,0.007792,0.007792,0.007792,0.007792,0.007792,0.007792,0.007792,0.007792,3.683969e-10,3.309546e-08,0,0.0


[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

## 01 — Qual é a escala do KPI, do gasto e da atividade de mídia?

### Pergunta
Qual é a escala do KPI, do gasto e da atividade de mídia?

### Motivação
Participações, assimetria, zeros e extremos ajudam a entender o desenho dos dados antes de comparar mercados.

### Método
Somar contagens e gastos por semana; calcular percentis, IQR, CV, fichas por canal, participação no gasto, sequências de atividade e de zeros. Sinalizar IQR 1,5× e |z modificado MAD| > 3,5, preservando observações.

### Interpretação antes de olhar os resultados
Share de gasto mede alocação. Um canal com poucos zeros não é necessariamente eficiente. IQR/MAD são triagem: escala populacional, intermitência e unidades podem explicar extremos.

**Dependências:** painel validado e funções em `src/eda.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [3]:
from src.eda import general
report_01 = general(geo_time_panel)
show(report_01)

### Resultado observado e conclusão parcial

O KPI nacional tem média semanal de 422,632,783 conversões sintéticas e CV de 6.9%. O agregado nacional soma conversões entre geos; não soma taxas.

O maior share de gasto é de Channel3 (40.0%). Share mede alocação, não retorno. A atividade varia de 35.7% a 97.3% das células GEO × semana. Exportamos todas as sequências de atividade e zeros, sem classificar com limiares arbitrários.

Foram sinalizados 6039 pares observação/variável por IQR (1,5×) ou MAD (|z modificado|>3,5). São regras de triagem, não critérios de remoção. Séries esparsas e geos grandes podem ser sinalizados legitimamente. Quando MAD é zero o escore fica indefinido. Nenhum extremo foi removido.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Dinâmica temporal dos sinais e seus controles.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**national_descriptive** — 15 linhas; CSV completo em `outputs/tables/`.

,n,mean,median,std,variance,min,max,p01,p05,p25,p50,p75,p95,p99,iqr,cv,zeros,zero_share
conversions,156.0,4.226328e+08,4.227034e+08,2.911830e+07,8.478753e+14,3.402833e+08,4.912874e+08,3.553039e+08,3.736850e+08,4.053507e+08,4.227034e+08,4.404787e+08,4.687004e+08,4.888518e+08,3.512797e+07,0.068897,0.0,0.000000
Channel0_impression,156.0,3.540654e+07,3.496160e+07,1.210007e+07,1.464118e+14,7.851189e+06,6.386554e+07,1.015168e+07,1.569336e+07,2.704481e+07,3.496160e+07,4.495542e+07,5.706419e+07,6.225383e+07,1.791061e+07,0.341747,0.0,0.000000
Channel1_impression,156.0,2.082422e+07,1.937102e+07,1.017229e+07,1.034756e+14,2.729719e+06,5.236681e+07,4.864523e+06,7.073937e+06,1.340884e+07,1.937102e+07,2.684337e+07,4.052737e+07,5.030103e+07,1.343453e+07,0.488484,0.0,0.000000
Channel2_impression,156.0,1.042020e+07,8.122188e+06,9.076909e+06,8.239028e+13,0.000000e+00,4.608557e+07,0.000000e+00,6.173670e+05,3.297471e+06,8.122188e+06,1.566935e+07,2.866586e+07,3.890450e+07,1.237188e+07,0.871088,4.0,0.025641
Channel3_impression,156.0,7.227242e+07,7.085189e+07,2.220926e+07,4.932513e+14,1.796734e+07,1.402645e+08,2.699388e+07,4.020701e+07,5.635578e+07,7.085189e+07,8.414181e+07,1.085364e+08,1.272496e+08,2.778603e+07,0.307299,0.0,0.000000
Channel4_impression,156.0,3.926712e+07,3.773983e+07,1.596923e+07,2.550163e+14,6.971125e+06,8.370790e+07,8.949558e+06,1.448040e+07,2.745158e+07,3.773983e+07,4.983927e+07,6.684530e+07,7.740700e+07,2.238769e+07,0.406682,0.0,0.000000
Channel0_spend,156.0,2.596264e+05,2.563638e+05,8.872649e+04,7.872390e+09,5.757060e+04,4.683083e+05,7.443948e+04,1.150750e+05,1.983121e+05,2.563638e+05,3.296456e+05,4.184359e+05,4.564901e+05,1.313335e+05,0.341747,0.0,0.000000
Channel1_spend,156.0,2.007711e+05,1.867605e+05,9.807345e+04,9.618401e+09,2.631785e+04,5.048806e+05,4.690000e+04,6.820147e+04,1.292777e+05,1.867605e+05,2.588031e+05,3.907338e+05,4.849639e+05,1.295254e+05,0.488484,0.0,0.000000
Channel2_spend,156.0,7.743173e+04,6.035536e+04,6.744983e+04,4.549479e+09,0.000000e+00,3.424584e+05,0.000000e+00,4.587608e+03,2.450326e+04,6.035536e+04,1.164378e+05,2.130138e+05,2.890964e+05,9.193452e+04,0.871088,4.0,0.025641
Channel3_spend,156.0,5.632074e+05,5.521375e+05,1.730732e+05,2.995434e+10,1.400167e+05,1.093059e+06,2.103590e+05,3.133268e+05,4.391716e+05,5.521375e+05,6.557037e+05,8.458071e+05,9.916360e+05,2.165321e+05,0.307299,0.0,0.000000


**channel_summary** — 5 linhas; CSV completo em `outputs/tables/`.

,total_media_units,total_spend,spend_share,active_periods,active_geos,cpmu,n,mean,median,std,...,p05,p25,p50,p75,p95,p99,iqr,cv,zeros,zero_share
channel,,,,,,,,,,,,,,,,,,,,,
Channel0,5523420467,4.050171e+07,0.184524,156,40,0.007333,6240,8.851635e+05,679670.5,8.192795e+05,...,0.0,253645.75,679670.5,1324174.25,2527659.25,3458103.74,1070528.50,0.925569,755,0.120994
Channel1,3248578573,3.132030e+07,0.142694,156,40,0.009641,6240,5.206055e+05,310652.5,6.205734e+05,...,0.0,0.00,310652.5,807552.50,1825045.65,2604638.15,807552.50,1.192022,1742,0.279167
Channel2,1625551369,1.207935e+07,0.055033,152,40,0.007431,6240,2.605050e+05,0.0,5.551397e+05,...,0.0,0.00,0.0,254734.00,1425716.35,2614612.82,254734.00,2.131013,4010,0.642628
Channel3,11274497204,8.786036e+07,0.400289,156,40,0.007793,6240,1.806810e+06,1505379.0,1.304714e+06,...,168567.8,808356.50,1505379.0,2566118.25,4282815.55,5741464.61,1757761.75,0.722109,169,0.027083
Channel4,6125670562,4.773068e+07,0.217459,156,40,0.007792,6240,9.816780e+05,746247.5,9.141036e+05,...,0.0,267594.50,746247.5,1476707.75,2788040.00,3776701.79,1209113.25,0.931164,820,0.131410


...

,active_cells,zero_cells,active_share,active_periods,active_geos
channel,,,,,
Channel0,5485,755,0.879006,156,40
Channel1,4498,1742,0.720833,156,40
Channel2,2230,4010,0.357372,152,40
Channel3,6071,169,0.972917,156,40
Channel4,5420,820,0.868590,156,40


[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

## 02 — Como o mercado e seus controles evoluem no tempo?

### Pergunta
Como o mercado e seus controles evoluem no tempo?

### Motivação
Choques comuns, tendência e sazonalidade podem criar associações que se parecem com resposta à mídia.

### Método
Séries nacionais e média móvel de 13 semanas, STL anual aproximado de 52 semanas, ACF, diferenças e picos. Para cada canal: mídia, gasto, CPMU e z-score versus KPI. Controles/Promo usam média ponderada por população; orgânico usa soma.

### Interpretação antes de olhar os resultados
Três ciclos anuais oferecem evidência limitada de sazonalidade. O z-score serve para comparar forma, não unidades ou tamanho de efeito. Mudanças de nível são descritivas, sem teste confirmatório de quebra.

**Dependências:** painel validado e funções em `src/eda.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [4]:
from src.eda import temporal
report_02 = temporal(geo_time_panel)
show(report_02)

### Resultado observado e conclusão parcial

O máximo nacional ocorre em 2023-05-15 (491,287,369) e o mínimo em 2022-09-19 (340,283,293). A média móvel de 13 semanas suaviza a série sem antecipar valores futuros; é apenas visual.

A decomposição STL usa período 52 como aproximação anual para dados semanais. Há apenas 3.0 ciclos: tendência, sazonalidade e mudanças de nível não são evidência confirmatória de mecanismos de negócio. Nenhum teste formal de quebra foi feito. As maiores mudanças e a ACF foram exportadas.

Todos os canais têm gráficos separados de impressões, gasto, CPMU e comparação padronizada com KPI. Controles e Promo são agregados por média ponderada por população porque somar índices sintéticos não tem interpretação. A mídia orgânica é somada. Existem 0 variáveis constantes entre os controles/tratamentos apresentados. Valores próximos de constantes devem ser avaliados na sua escala, sem corte universal.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Heterogeneidade entre geos e decomposição within/between.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**control_temporal_summary** — 4 linhas; CSV completo em `outputs/tables/`.

,variable,unique,range,n,mean,median,std,variance,min,max,...,p05,p25,p50,p75,p95,p99,iqr,cv,zeros,zero_share
0,competitor_sales_control,6240,8.750317e+00,6240,-0.044412,-0.050346,1.210067,1.464262e+00,-4.825634,3.924682e+00,...,-1.995962,-0.872736,-0.050346,0.743633,1.987068e+00,2.869632e+00,1.616369,-27.246396,0,0.000000
1,sentiment_score_control,6239,8.550651e+00,6240,-0.033675,-0.036433,1.175478,1.381749e+00,-4.230392,4.320259e+00,...,-1.940917,-0.808085,-0.036433,0.739391,1.891985e+00,2.756426e+00,1.547475,-34.906452,0,0.000000
2,Promo,3319,4.800555e+00,6240,0.492528,0.084382,0.694224,4.819467e-01,0.000000,4.800555e+00,...,0.000000,0.000000,0.084382,0.832968,1.962645e+00,2.697505e+00,0.832968,1.409511,2922,0.468269
3,Organic_channel0_impression,4164,5.801781e+06,6240,532466.670513,269958.000000,696134.901248,4.846038e+11,0.000000,5.801781e+06,...,0.000000,0.000000,269958.000000,799699.250000,1.971783e+06,3.005490e+06,799699.250000,1.307377,2075,0.332532


**national_timeseries** — 156 linhas; CSV completo em `outputs/tables/`.

,conversions,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,Channel4_impression,Channel0_spend,Channel1_spend,Channel2_spend,Channel3_spend,Channel4_spend,Organic_channel0_impression,competitor_sales_control,sentiment_score_control,Promo,revenue_derived,revenue_per_conversion,population
time,,,,,,,,,,,,,,,,,,
2021-01-25,3.863009e+08,44340630,10537596,6501889,66738506,53026011,325137.568300,101595.404270,48315.045180,520082.52420,413173.984050,24028870,-0.032438,-0.342020,0.272211,7.731975e+06,0.020015,21696753.31
2021-02-01,3.553039e+08,47391940,30857600,7966320,79000018,27592214,347511.968090,297505.263890,59197.121480,615634.54030,214996.093200,29188267,0.312002,0.807150,0.151264,7.104532e+06,0.019996,21696753.31
2021-02-08,4.383089e+08,28960185,23144749,4637739,68442555,20057461,212357.019450,223143.885196,34462.687700,533361.91650,156285.964472,12051369,-1.243697,-0.152005,0.596395,8.762589e+06,0.019992,21696753.31
2021-02-15,4.186124e+08,38126016,10667077,3029637,70601091,37246623,279567.522900,102843.758830,22513.003350,550183.03800,290222.395240,9595212,-0.019225,-0.377523,0.341224,8.363356e+06,0.019979,21696753.31
2021-02-22,3.557853e+08,46890794,17925961,10486840,81842660,56259530,343837.209200,172828.342330,77926.915300,637786.80240,438369.280950,9288846,0.344244,0.034181,0.366063,7.109799e+06,0.019983,21696753.31
2021-03-01,4.406152e+08,31496378,19099596,3713252,58861970,26886668,230954.220330,184143.627600,27592.895000,458701.94280,209498.541240,13711117,-0.691891,-0.554818,0.371434,8.813679e+06,0.020003,21696753.31
2021-03-08,4.530552e+08,25772939,4734654,7271164,68725846,37615253,188985.826030,45647.895670,54031.470175,535569.55042,293094.728570,13763654,-0.848894,-0.373470,0.091544,9.050774e+06,0.019977,21696753.31
2021-03-15,3.905958e+08,35435485,17853348,8084840,73628817,40915556,259838.598400,172128.262440,60077.836005,573777.62510,318810.394260,17933987,0.454243,0.151022,0.012450,7.804657e+06,0.019981,21696753.31
2021-03-22,3.790501e+08,28273144,27489089,6237490,64406105,56582012,207319.137090,265028.672380,46350.316360,501906.50854,440882.035900,31799726,0.560951,-1.544945,0.928034,7.584097e+06,0.020008,21696753.31


...

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

## 03 — As diferenças entre geos vão além do tamanho populacional?

### Pergunta
As diferenças entre geos vão além do tamanho populacional?

### Motivação
Um painel pode ter muitas linhas e ainda ser formado por cópias escaladas da série nacional. Precisamos separar tamanho, diferenças persistentes e diferenças de dinâmica.

### Método
Rankings e concentração do KPI, população, KPI semanal por 1.000 habitantes, sincronização, small multiples selecionados por critério. Decompor variância total = between + within; calcular R²_geo, R²_time e resíduo two-way em escala bruta e per capita.

### Interpretação antes de olhar os resultados
R²_geo alto indica diferenças persistentes. R²_time alto indica movimento comum. Within inclui movimento nacional, logo não equivale à informação adicional geo-temporal. No painel balanceado: R²_geo + R²_time + fração residual = 1. Essa identidade não demonstra causalidade.

**Dependências:** painel validado e funções em `src/geo_analysis.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [5]:
from src.geo_analysis import geo_analysis
report_03 = geo_analysis(geo_time_panel)
show(report_03)

### Resultado observado e conclusão parcial

O maior geo responde por 5.0% do KPI acumulado; os cinco maiores por 22.9%. A correlação populacional com KPI total é 0.978 (Spearman). A escala populacional precisa ser considerada antes de interpretar rankings.

O KPI tem 67.9% da variância between-geo e 32.1% within-geo. A identidade usa momentos populacionais (ddof=0), com peso igual por observação. R²_geo equivale à parcela between; R²_time quantifica semanas comuns. No painel balanceado, a parcela residual two-way do KPI é 30.7%; por habitante é 86.1%.

Within não significa automaticamente sinal geográfico novo: inclui choques temporais nacionais. Por isso apresentamos também o resíduo após remover simultaneamente geo e tempo. Essa decomposição é descritiva e depende da escala. Os small multiples usam geos escolhidos por critérios registrados; os cálculos e heatmaps incluem todos os geos. Geo0…Geo39 são rótulos sintéticos, sem fronteiras para um mapa geográfico real.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Mix, alocação territorial, concentração e variação de cada canal dentro dos geos.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**geo_summary** — 40 linhas; CSV completo em `outputs/tables/`.

,kpi_total,mean,std,min,max,population,kpi_share,population_share,kpi_per_1000_per_week
geo,,,,,,,,,
Geo36,3.290565e+09,2.109336e+07,5.507861e+06,4166830.8,36644916.0,969886.50,0.049909,0.044702,21748.281403
Geo39,3.222847e+09,2.065927e+07,4.438007e+06,6550282.0,30060602.0,798939.10,0.048882,0.036823,25858.383877
Geo10,3.093954e+09,1.983304e+07,6.371937e+06,5597731.5,33824700.0,994048.94,0.046927,0.045816,19951.772822
Geo23,2.836450e+09,1.818237e+07,5.226278e+06,5906972.0,36712692.0,981570.90,0.043022,0.045240,18523.746607
Geo31,2.684670e+09,1.720942e+07,4.773290e+06,5595800.0,31523720.0,873369.70,0.040720,0.040253,19704.623992
Geo7,2.578130e+09,1.652648e+07,4.180518e+06,5721620.0,28238984.0,733989.56,0.039104,0.033829,22515.954581
Geo2,2.468820e+09,1.582577e+07,4.684044e+06,4341767.0,29242514.0,892369.50,0.037446,0.041129,17734.548771
Geo25,2.437115e+09,1.562253e+07,4.305033e+06,4081507.0,25940226.0,779181.90,0.036965,0.035912,20049.916609
Geo17,2.201843e+09,1.411438e+07,4.340580e+06,3393230.5,25439212.0,777539.94,0.033396,0.035837,18152.606923


**decomposition** — 15 linhas; CSV completo em `outputs/tables/`.

,total_variance,between_geo,within_geo,between_share,within_share,r2_geo,r2_time,r2_geo_time,residual_geo_time_share
variable,,,,,,,,,
conversions,3.695331e+13,2.508695e+13,1.186637e+13,0.678882,0.321118,0.678882,0.014340,0.693223,0.306777
Channel0_impression,6.712189e+11,1.667112e+11,5.045076e+11,0.248371,0.751629,0.248371,0.136330,0.384701,0.615299
Channel1_impression,3.851114e+11,5.549878e+10,3.296126e+11,0.144111,0.855889,0.144111,0.167931,0.312042,0.687958
Channel2_impression,3.081801e+11,1.648447e+10,2.916956e+11,0.053490,0.946510,0.053490,0.167090,0.220580,0.779420
Channel3_impression,1.702277e+12,6.636814e+11,1.038596e+12,0.389879,0.610121,0.389879,0.181100,0.570978,0.429022
Channel4_impression,8.355853e+11,1.982119e+11,6.373734e+11,0.237213,0.762787,0.237213,0.190747,0.427960,0.572040
Channel0_spend,3.609065e+07,8.963869e+06,2.712678e+07,0.248371,0.751629,0.248371,0.136330,0.384701,0.615299
Channel1_spend,3.579739e+07,5.158797e+06,3.063859e+07,0.144111,0.855889,0.144111,0.167931,0.312042,0.687958
Channel2_spend,1.701728e+07,9.102497e+05,1.610703e+07,0.053490,0.946510,0.053490,0.167090,0.220580,0.779420


...

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

## 04 — Como e onde os canais recebem investimento?

### Pergunta
Como e onde os canais recebem investimento?

### Motivação
O valor do painel para MMM depende também de execução territorial e temporal diferenciada, e não somente do número de geos.

### Método
Matrizes de gasto, mix dentro de cada geo e alocação de cada canal; intensidade por habitante; Spearman/Pearson com população; HHI; CV within; heatmaps GEO × semana para gasto e impressões, brutos e per capita.

### Interpretação antes de olhar os resultados
Mix tem denominador por geo; alocação tem denominador por canal. HHI mede concentração, não performance. Diferenças de cor entre figuras não são comparáveis sem observar escalas. Geos seguem ordem fixa pelo KPI total.

**Dependências:** painel validado e funções em `src/geo_analysis.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [6]:
from src.geo_analysis import media_geo
report_04 = media_geo(geo_time_panel)
show(report_04)

### Resultado observado e conclusão parcial

O mix divide cada linha GEO × canal pelo orçamento total do geo; a alocação divide cada coluna pelo orçamento do canal. Essas perguntas são diferentes. A maior amplitude de share do mix é 3.7%, em Channel1.

HHI é a soma dos quadrados das participações geográficas de um canal: varia de 1/G (uniforme) a 1 (um único geo). Observamos 0.030 a 0.031, sem aplicar limiares de concorrência econômica. O inverso traduz concentração em número equivalente de geos uniformes, não em tamanho amostral efetivo.

Os CVs within variam de 0.442 a 2.153. Médias zero produzem CV indefinido. Heatmaps usam a mesma ordenação de geos por KPI total e escalas de cor próprias por variável; não comparar cores entre painéis sem ler as escalas. As versões por habitante ajudam a investigar diferenças que persistem além do tamanho dos mercados.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Correlações entre canais e controles, VIF e conflitos com geo/tempo.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**media_mix** — 40 linhas; CSV completo em `outputs/tables/`.

channel,Channel0,Channel1,Channel2,Channel3,Channel4
geo,,,,,
Geo36,0.190032,0.145227,0.061584,0.389280,0.213879
Geo39,0.198288,0.138034,0.048407,0.387838,0.227433
Geo10,0.177915,0.137524,0.049388,0.411376,0.223797
Geo23,0.184789,0.147995,0.059540,0.401638,0.206038
Geo31,0.199994,0.154922,0.050278,0.380017,0.214790
Geo7,0.189596,0.135009,0.060125,0.393955,0.221316
Geo2,0.179638,0.139828,0.061873,0.406928,0.211733
Geo25,0.192168,0.149787,0.049639,0.394906,0.213501
Geo17,0.195987,0.152988,0.046717,0.389405,0.214902


**geo_allocation** — 40 linhas; CSV completo em `outputs/tables/`.

channel,Channel0,Channel1,Channel2,Channel3,Channel4
geo,,,,,
Geo36,0.045798,0.045260,0.049764,0.043248,0.043739
Geo39,0.038196,0.034384,0.031265,0.034439,0.037175
Geo10,0.042522,0.042504,0.039579,0.045324,0.045387
Geo23,0.045338,0.046955,0.048981,0.045426,0.042895
Geo31,0.042802,0.042875,0.036079,0.037491,0.039006
Geo7,0.036130,0.033270,0.038417,0.034607,0.035787
Geo2,0.041837,0.042112,0.048317,0.043688,0.041844
Geo25,0.037912,0.038214,0.032836,0.035915,0.035742
Geo17,0.038819,0.039185,0.031025,0.035554,0.036118


...

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

## 05 — Quais sinais se movem juntos ou podem ser redundantes?

### Pergunta
Quais sinais se movem juntos ou podem ser redundantes?

### Motivação
Mídia pode responder à demanda e compartilhar tendência, sazonalidade ou alocação com outros canais. Isso limita interpretações ingênuas.

### Método
Pearson/Spearman nacional, overall, within e two-way; matrizes separadas de gasto e mídia; VIF com intercepto em preditores, sem duplicar spend e impressões; lags de 0 a 8 semanas, com contagem das observações alinhadas.

### Interpretação antes de olhar os resultados
Não usar VIF como regra automática de remoção. Correlações entre gasto e impressões podem ser contábeis. Lags não estimam adstock. Não calcular significância supondo independência de linhas geo-temporais.

**Dependências:** painel validado e funções em `src/eda.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [7]:
from src.eda import relationships
report_05 = relationships(geo_time_panel)
show(report_05)

### Resultado observado e conclusão parcial

Exportamos Pearson e Spearman em quatro níveis: nacional, painel bruto, within-geo e residual two-way. Within remove a média de cada geo; two-way remove também a média semanal comum. Spearman de resíduos mede a ordenação dos resíduos e não é uma correlação parcial de postos.

O maior VIF within entre preditores é 2.11. A regressão auxiliar inclui intercepto e colunas padronizadas. Spend e impressões são diagnosticados em matrizes separadas; colocá-los juntos no VIF duplicaria sinais de custo praticamente fixo. Constantes são indefinidas e dependência linear perfeita gera infinito.

A maior associação absoluta within entre todas as variáveis comparadas é Channel1_impression × Channel1_spend (1.000); isso pode ser uma relação contábil entre gasto e mídia. Lags 0–8 são exploratórios, sem selecionar adstock. Não calculamos p-valores que presumiriam independência das 6,240 linhas: existe dependência temporal e geográfica.

Diferenças entre correlações nacionais e within podem resultar de composição populacional, tendência, sazonalidade e mídia alocada em antecipação à demanda. Correlação mídia–KPI não constitui efeito de mídia.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Checagens independentes da ferramenta oficial e comparação de definições.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**vif** — 36 linhas; CSV completo em `outputs/tables/`.

,variable,level,vif,status
0,Channel0_impression,national,1.719037,finite
1,Channel1_impression,national,1.506175,finite
2,Channel2_impression,national,2.890948,finite
3,Channel3_impression,national,3.857286,finite
4,Channel4_impression,national,2.563407,finite
5,Organic_channel0_impression,national,2.376850,finite
6,competitor_sales_control,national,3.793200,finite
7,sentiment_score_control,national,3.461720,finite
8,Promo,national,1.782877,finite
9,Channel0_impression,overall,1.632356,finite


**correlations** — 15 linhas; CSV completo em `outputs/tables/`.

,level,method,left,right,correlation
451,within,pearson,Channel1_impression,Channel1_spend,1.000000
463,within,pearson,Channel2_impression,Channel2_spend,1.000000
484,within,pearson,Channel4_impression,Channel4_spend,1.000000
474,within,pearson,Channel3_impression,Channel3_spend,1.000000
438,within,pearson,Channel0_impression,Channel0_spend,1.000000
486,within,pearson,Channel4_impression,competitor_sales_control,0.550356
516,within,pearson,Channel4_spend,competitor_sales_control,0.550356
478,within,pearson,Channel3_impression,sentiment_score_control,0.506064
513,within,pearson,Channel3_spend,sentiment_score_control,0.506064
459,within,pearson,Channel2_impression,Channel3_impression,0.503550


...

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

## 06 — A auditoria oficial encontra os mesmos padrões?

### Pergunta
A auditoria oficial encontra os mesmos padrões?

### Motivação
Uma implementação independente pode revelar diferenças de definição, escalonamento e critérios de alerta.

### Método
Construir InputData com DataFrameInputDataBuilder; instanciar Meridian com defaults provisórios e MeridianEDA; gerar HTML, checks, warnings e artefatos numéricos. Recalcular R² ajustado na escala oficial por soma de quadrados e comparar com tolerância 1e-5.

### Interpretação antes de olhar os resultados
A API 2.1.0 gera amostras PRIOR automaticamente; não há posterior ou MCMC. Os defaults não são especificação final. INFO não significa ausência de problema; REVIEW e FAIL precisam ser lidos. R² oficial é ajustado e usa variáveis transformadas.

**Dependências:** painel validado e funções em `src/official_eda.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [8]:
from src.official_eda import official_eda
report_06 = official_eda(geo_time_panel)
show(report_06)

I0000 00:00:1791024430.472271    2646 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791024430.472744    2646 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1791024431.526142    2646 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791024431.526422    2646 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### Resultado observado e conclusão parcial

O relatório HTML oficial foi executado com Meridian 2.1.0, sem amostragem posterior. Os checks exportaram 16 achados: {'INFO': 10, 'REVIEW': 6}. As advertências de execução estão em meridian_warnings.csv; não foram silenciadas nem resolvidas por alteração de thresholds.

A API atual usa DataFrameInputDataBuilder. MeridianEDA gera amostras da prior no construtor para completar diagnósticos da especificação. Isso não ajusta o modelo aos dados e não produz estimativas posteriores. A especificação default é provisória: 156 knots e 8 lags. Não escolhemos hiperparâmetros finais com esta EDA.

A razão oficial é 6240/(40−1+156+2+7) = 30.59. Essa contagem é um guardrail que não conta todos os efeitos geo como parâmetros independentes. Não é prova de tamanho amostral efetivo ou de suficiência.

Os alertas de CPMU precisam de contexto: o CV máximo do custo unitário é 3.475e-08. A variação relativa minúscula é compatível com arredondamento na geração/exportação, e não evidencia dispersão econômica de custos. Não alteramos o dado ou o threshold IQR oficial para eliminar o alerta.

Uma diferença metodológica importante: os R² oficiais são ajustados e usam dados transformados; a EDA própria mostra R² brutos e por habitante não ajustados. Recalculando por soma de quadrados na mesma escala oficial, a concordância dentro de 1e-5 foi **True**. A tabela official_r2_reconciliation contém as diferenças por variável. VIF e correlações também dependem da população e agregação. Artefatos numéricos foram exportados para investigar essas diferenças.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Consolidar os achados, implicações e limites de prontidão.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**official_checks** — 16 linhas; CSV completo em `outputs/tables/`.

,check,severity,cause,explanation
0,check_geo_cost_per_media_unit,REVIEW,OUTLIER,There are outliers in cost per media unit acro...
1,check_national_cost_per_media_unit,REVIEW,OUTLIER,There are outliers in cost per media unit acro...
2,check_geo_std,REVIEW,OUTLIER,There are outliers in the scaled KPI in certai...
3,check_geo_std,REVIEW,OUTLIER,There are outliers in the scaled treatment or ...
4,check_national_std,REVIEW,OUTLIER,There are outliers in the scaled KPI. Please c...
5,check_national_std,REVIEW,OUTLIER,There are outliers in the scaled treatment or ...
6,check_overall_kpi_invariability,INFO,NONE,The kpi_scaled has variability across geos and...
7,check_population_corr_raw_media,INFO,NONE,Please review the Spearman correlation between...
8,check_population_corr_scaled_treatment_control,INFO,NONE,Please review the Spearman correlation between...
9,check_geo_pairwise_corr,INFO,NONE,Please review the computed pairwise correlatio...


**data_adequacy** — 1 linhas; CSV completo em `outputs/tables/`.

,n_geos,n_times,n_knots,n_controls,n_treatments,n_data_points,n_parameters,ratio
0,40,156,156,2,7,6240,204,30.588235


**comparison** — 7 linhas; CSV completo em `outputs/tables/`.

,issue,custom_eda,custom_evidence,meridian_eda,agreement,interpretation
0,cost_per_media_unit,CPMU = spend/media; NaN em zero,cost_per_media_unit.csv,{'REVIEW': 2},"REVIEW por extremos, mas CV máximo de CPMU=3.4...",Mesmas unidades; verificar artefatos por geo e...
1,std,"Variância bruta, zeros, IQR e MAD",channel_summary.csv,{'REVIEW': 4},comparação de definições; não igualdade numéri...,Escalas distintas: Meridian transforma as vari...
2,population,Spearman de população × total de mídia,population_media_correlation.csv,{'INFO': 2},comparação de definições; não igualdade numéri...,Total e média temporal são proporcionais em pa...
3,pairwise,"Pearson/Spearman nacional, overall e within",correlation_summary.csv,{'INFO': 2},comparação de definições; não igualdade numéri...,Meridian usa transformação própria; within da ...
4,vif,VIF com intercepto em preditores brutos/within,vif_summary.csv,{'INFO': 2},comparação de definições; não igualdade numéri...,Diagnósticos oficiais usam escalas/população; ...
5,geo_time,R² não ajustado em valores brutos e per capita,geo_time_r2.csv,{'INFO': 2},R² ajustado reconciliado na mesma escala: True...,Meridian usa R² ajustado sobre variáveis trans...
6,data_param,G×T observações; parâmetros não são todos inde...,data_audit.csv,{'INFO': 1},comparação de definições; não igualdade numéri...,Guardrail provisório; razão não prova adequaçã...


In [9]:
from IPython.display import display, FileLink
display(FileLink("outputs/reports/meridian_eda_report.html"))
# Baixe o HTML e abra no navegador para explorar os gráficos oficiais.

/workspace/scratch/5d823022ed95/tcc-meridian/outputs/reports/meridian_eda_report.html

## 07 — Há evidência para justificar investigar um MMM geo-level?

### Pergunta
Há evidência para justificar investigar um MMM geo-level?

### Motivação
Precisamos responder às perguntas do TCC com resultados, limites e implicações, evitando transformar descrição em causalidade.

### Método
Consolidar integridade, missing, extremos, variabilidade, população, mix, within/between, R², colinearidade e adequação oficial; produzir relatório de prontidão e narrativa final.

### Interpretação antes de olhar os resultados
Informação geo-temporal residual pode justificar a investigação hierárquica, mas não garante causalidade, generalização ou superioridade preditiva. A próxima fase requer especificação, priors, holdout e diagnósticos posteriores.

**Dependências:** painel validado e funções em `src/reporting.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [10]:
from src.reporting import conclusions
report_07 = conclusions(geo_time_panel)
show(report_07)

### Resultado observado e conclusão parcial

1. **Estrutura.** O painel tem 40 geos × 156 semanas = 6240 observações, completude de 100.0% e 0 células ausentes. A chave geo–tempo foi validada.

2. **Tempo.** Todos os 5 canais pagos variam; a parcela within-geo da variância fica entre 61.0% e 94.7%. Channel2 é o canal com mais zeros: 64.3% das células, com 5.5% do gasto. A tabela de atividade e os gráficos distinguem semanas sem mídia de mudanças de intensidade. Movimentos de nível e sazonalidade são exploratórios.

3. **Geografia.** As médias por geo explicam de 5.3% a 39.0% da variância bruta da mídia paga. Rankings de volume dependem de população; mix e medidas por habitante oferecem lentes complementares.

4. **GEO × TEMPO.** Após retirar médias de geo e de semana, permanece de 42.9% a 77.9% da variância bruta de mídia paga; por habitante, de 64.8% a 79.3%. Esse componente é perdido ao observar apenas o agregado nacional. O painel contém variação potencialmente informativa, mas o resíduo pode conter ruído, endogeneidade e confundimento. Comparar com a decomposição per capita é essencial: diferenças multiplicativas de tamanho também podem gerar resíduos aditivos.

5. **Identificabilidade.** R², correlações, VIF e os alertas oficiais devem orientar hipóteses de especificação. Nenhuma dessas métricas prova identificação causal ou que um modelo geo terá melhor previsão que o nacional. Essa comparação exige futura modelagem com validação temporal comum, priors justificados e diagnósticos posteriores.

**Prontidão:** a integridade permite continuar o estudo metodológico, condicionando a próxima fase à revisão dos alertas oficiais e das limitações dos dados simulados. Não há recomendação de alocação de orçamento nesta EDA.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Desenho causal, especificação, priors, holdout temporal e posterior em fase futura.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**readiness_report** — 18 linhas; CSV completo em `outputs/tables/`.

,Dimension,Question,Evidence,Finding,Modeling_implication
0,integridade,Painel completo?,data_audit,"6240 linhas, 40 geos, 156 semanas; completude 1.0",Contrato validado; preservar raw e schema.
1,missing,Há valores ausentes?,missing_summary,0 células ausentes,Não imputar dados completos.
2,outliers,Há extremos descritivos?,outliers,6039 pares observação/variável sinalizados por...,Investigar escala populacional e zeros; não ex...
3,KPI variation,Existe variação?,within_between_variance,1/1 variáveis com variância positiva; resíduo ...,"Variação é necessária, não suficiente para ide..."
4,media variation,Existe variação?,within_between_variance,6/6 variáveis com variância positiva; resíduo ...,"Variação é necessária, não suficiente para ide..."
5,spend variation,Existe variação?,within_between_variance,5/5 variáveis com variância positiva; resíduo ...,"Variação é necessária, não suficiente para ide..."
6,control collinearity,Existe variação?,within_between_variance,2/2 variáveis com variância positiva; resíduo ...,"Variação é necessária, não suficiente para ide..."
7,temporal variation,Quanto varia a mídia paga?,geo_time_r2,r2_time: 13.63% a 19.07%,Avaliar também escala por população e colinear...
8,geo variation,Quanto varia a mídia paga?,geo_time_r2,r2_geo: 5.35% a 38.99%,Avaliar também escala por população e colinear...
9,within variation,Quanto varia a mídia paga?,geo_time_r2,within_share: 61.01% a 94.65%,Avaliar também escala por população e colinear...


[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

## Levar os resultados do Colab
A célula abaixo prepara um ZIP com tabelas, relatórios, figuras e metadados. No Colab, o download exige somente descomentar as duas linhas indicadas.

In [11]:
import zipfile
archive_path = ROOT / 'EDA_RESULTS.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in ['outputs', 'docs', 'data/metadata']:
        for path in (ROOT / folder).rglob('*'):
            if path.is_file() and '.mplcache' not in str(path):
                archive.write(path, path.relative_to(ROOT))
print(archive_path)
# from google.colab import files
# files.download(str(archive_path))


/workspace/scratch/5d823022ed95/tcc-meridian/EDA_RESULTS.zip
